# Имагология · блокнот 2. Карта взгляда

| | |
|---|---|
| **Вопрос** | Какое пространство строит травелог: что названо, что пропущено, где граница своего? |
| **Что вы получите** | Интерактивную карту маршрутов и мест текста, таблицу мест, GeoJSON для QGIS или geojson.io, список неизвестных имён. |
| **Стенд в браузере** | [Карта взгляда](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/karta.html) |
| **Практические** | 4, 8 · [задания](https://pavelaleks.github.io/samarkand_lectures/imagology/lab/practicum.html) |

**Правило курса:** описываем образ в тексте, а не «правду» о народе. Число без контекста в работу не идёт: каждую цифру проверьте предложением.

### Как работать
1. **Файл → Сохранить копию на Диске**, иначе правки пропадут.
2. Запускайте ячейки **сверху вниз** кнопкой ▶. Код спрятан, меняются только **поля формы** справа.
3. Сначала запустите всё как есть: демо на готовой паре текстов из Викитеки. Потом впишите свои тексты.

**Как задать тексты** (в полях через `;`):
- ключ готового текста: `pushkin-arzrum`, `custine`, `vambery` — список выводит `list_texts()`;
- страница Викитеки: `ws:ru:Мцыри (Лермонтов)`, `ws:en:The Giaour`, `ws:fr:Les Orientales`;
- ссылка на .txt: `https://www.gutenberg.org/cache/epub/41751/pg41751.txt`;
- `МОИ ФАЙЛЫ` — появится кнопка загрузки .txt или .docx.

> Ошибка? **Среда выполнения → Перезапустить сеанс**, затем ячейки сверху вниз. Викитека иногда отвечает медленно: подождите и запустите ячейку ещё раз.

In [ ]:
#@title 🔧 Подготовка — просто запустите ▶ { display-mode: "form" }
import subprocess, sys, importlib, requests
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "python-docx", "folium"], check=False)
_u = "https://pavelaleks.github.io/samarkand_lectures/imagology/lab/colab/imago_utils.py"
open("imago_utils.py", "wb").write(requests.get(_u, timeout=60).content)
import imago_utils; importlib.reload(imago_utils)
from imago_utils import *
setup()

In [ ]:
#@title 📂 Тексты { display-mode: "form" }
#@markdown Готовая пара заполнит оба корпуса (поля ниже тогда не нужны). Список пар — `list_pairs()`.
PAIR = "arzrum"  #@param ["—", "persidskie", "custine", "orientales", "divan", "itineraire", "arzrum", "plennik", "bela", "peri", "prorok", "spor", "ammalat", "burden", "samarkand", "sarty", "nikitin"]
#@markdown Корпус A (обычно Россия) и корпус B (обычно Запад): ключи, `ws:язык:Название`, ссылки, `МОИ ФАЙЛЫ` — через «;».
CORPUS_A = "pushkin-arzrum"  #@param {type:"string"}
CORPUS_B = "dumas-caucase"  #@param {type:"string"}

if PAIR != "—":
    _p = next(p for p in imago_utils._data("presets")["pairs"] if p["id"] == PAIR)
    CORPUS_A, CORPUS_B = _p["left"], _p["right"]
    print("Пара:", _p["title"]); print("Вопрос:", _p["question"])
A = analyze_all(load_spec(CORPUS_A, "россия")) if CORPUS_A.strip() else []
B = analyze_all(load_spec(CORPUS_B, "запад")) if CORPUS_B.strip() else []
print("\nКорпус A:", "; ".join(a["label"] for a in A)); print("Корпус B:", "; ".join(b["label"] for b in B))

In [ ]:
#@title 🧭 Маршруты { display-mode: "form" }
#@markdown Ключи через запятую: nikitin, karamzin, pushkin, lermontov, custine, dumas, vambery, vereshchagin, schuyler, burnaby.
ROUTES = "pushkin, dumas"  #@param {type:"string"}
_r = tuple(x.strip() for x in ROUTES.split(",") if x.strip())
list_routes()

In [ ]:
#@title 🗺️ Карта { display-mode: "form" }
M = make_map([c for c in (A, B) if c], routes=_r)
M.save("karta_vzglyada.html")
M

In [ ]:
#@title 📋 Места и регионы { display-mode: "form" }
PT = place_table(A + B)
print(PT.groupby(["текст", "регион"])["упоминаний"].sum().unstack(fill_value=0).T)
PT.head(40)

In [ ]:
#@title ❓ Неизвестные имена (кандидаты в места) { display-mode: "form" }
#@markdown Слова с заглавной буквы не в начале предложения, которых нет в газеттире. Проверьте вручную: место, человек или народ?
for w, n in unknown_names(A + B, 40):
    print(f"{n:4d}  {w}")

In [ ]:
#@title 💾 Сохранить GeoJSON, таблицу, карту { display-mode: "form" }
to_geojson([c for c in (A, B) if c], routes=_r)
save_excel({"Места": PT}, "mesta.xlsx")
for f in ("karta_vzglyada.geojson", "mesta.xlsx", "karta_vzglyada.html"):
    download(f)